In [42]:
import pandas as pd
print(pd.__version__)

2.3.3


In [43]:
df = pd.read_csv("../data/SMSSpamCollection", sep="\t", header=None, names=["label", "text"])
df.shape
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [44]:
df = pd.read_csv("../data/SMSSpamCollection", sep="\t", header=None,
                 names=["label", "text"], quoting=3)
df.shape

(5574, 2)

In [45]:
df.shape
df.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [46]:
df["label"].value_counts()
df["label"].value_counts(normalize=True)

label
ham     0.865985
spam    0.134015
Name: proportion, dtype: float64

In [47]:
df["length"] = df["text"].str.len()

In [48]:
df.groupby("label")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4827.0,71.471929,58.326643,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.676037,28.871250,13.0,133.0,149.0,157.0,223.0


In [51]:
df.groupby("label")["length"].mean()

label
ham      71.471929
spam    138.676037
Name: length, dtype: float64

In [58]:
df[df["label"] == "spam"].sample(10, random_state=42)


,label,text,length
1456,spam,Summers finally here! Fancy a chat or flirt wi...,159
1853,spam,This is the 2nd time we have tried 2 contact u...,154
673,spam,Get ur 1st RINGTONE FREE NOW! Reply to this ms...,153
947,spam,Ur cash-balance is currently 500 pounds - to m...,137
2881,spam,Last Chance! Claim ur £150 worth of discount v...,152
3642,spam,"You can stop further club tones by replying ""S...",155
1653,spam,For ur chance to win a £250 cash every wk TXT:...,122
1205,spam,WIN a year supply of CDs 4 a store of ur choic...,145
3968,spam,YOU HAVE WON! As a valued Vodafone customer ou...,134
2691,spam,sports fans - get the latest sports news str* ...,153


In [56]:
df.duplicated().sum()

np.int64(403)

In [57]:
df.isna().sum()

label     0
text      0
length    0
dtype: int64

## Knowledge Check 1

**1. How much of the data is spam? Is it balanced?**

About 13.4% spam (747) and 86.6% ham (4,827), 5,574 total. Not balanced, there's way more ham.

**2. What accuracy would "always predict ham" get?**
About 86.6%, and it would catch 0 scams. So accuracy looks great but tells me nothing here. I need recall and precision instead (TP, FP, FN), since I care about how many scams I actually catch.

**3. Which mistake is worse?**
A scam marked ham (FN) is worse. That's the one that gets them to give out their info. So I want to push FN down to drive recall up, since recall = how many real scams I caught. I still want precision decent so it doesn't flag everything, but recall comes first. (Tradeoff: higher recall usually means more false alarms.)

**4. Differences between spam and ham**
- Length: spam is longer on average (~139 vs ~71 chars) and way more consistent (most spam is 133-157 chars). Ham is all over the place (std ~58, max 910).
- Text patterns:
  1. Spam has numbers in it (phone numbers, prices, short codes). Ham mostly doesn't.
  2. Spam tells you to do something (call, text, claim, reply). Ham reads like normal conversation.

## Extra:
- Theres 403 duplicate messages (~7% of the data): 309 ham, 94 spam
- By rate, spam repeats more (about 12.6% of spam vs 6.4% of ham). Spammers send the same template to everyone. Ham repeats are short stuff like "Ok."
- Why this matters: if the same message is in both train and test, the model gets credit for remembering instead of learning. That's data leakage, and it makes my test score look better than it really is.


In [60]:
df[df.duplicated()]["label"].value_counts()

label
ham     309
spam     94
Name: count, dtype: int64

In [62]:
df["has_number"] = df["text"].str.contains(r"\d{5,}")   # 5+ digits in a row
df.groupby("label")["has_number"].mean()

label
ham     0.000622
spam    0.783133
Name: has_number, dtype: float64

Quick Note: Numbers: 78% of spam has a 5+ digit run (phone numbers, short codes, prices) vs 0.06% of ham.